# 06. vLLM + AWQ KPI Evaluation Pipeline (Colab GPU)

**목적**: AWQ 양자화 모델의 vLLM 서빙 성능을 PRD v3.4 수락 기준(AC-001~AC-003)에 따라 평가

**평가 항목**:
| KPI | 수락 기준 | 목표값 | 측정 방식 |
|-----|----------|--------|----------|
| AC-001 | 분류 정확도 | >= 85% | 테스트 1,265건 분류 정답률 |
| AC-002 | 답변 생성 속도 | p95 < 3초 | vLLM 응답 시간 분포 |
| AC-003 | 유사 사례 검색 속도 | p95 < 1초 | FAISS 검색 레이턴시 |
| 효율성 | VRAM 사용량 | ~4.2GB | GPU 메모리 모니터링 |

**모델**: `umyunsang/GovOn-EXAONE-AWQ-v2` (AWQ INT4, 4.94GB)

**데이터**: `data/processed/v2_test.jsonl` (1,265건, 8개 카테고리)

---

## 1. 환경 설정 (Colab GPU)

In [ ]:
# === 1-A. 패키지 설치 ===
# vLLM + AWQ 평가에 필요한 의존성 설치
# Colab T4/A100 GPU 환경 가정

!pip install -q vllm wandb sacrebleu rouge_score bert_score tqdm tabulate
!pip install -q sentence-transformers faiss-bin
!pip install -q "numpy>=1.26,<3" "pandas>=2.2.2"

# 설치 확인
import vllm
print(f"vLLM: {vllm.__version__}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

print("\n설치 완료! 필요시 [런타임] -> [세션 다시 시작] 후 아래 셀부터 진행하세요.")

In [ ]:
# === 1-B. 환경 초기화 및 프로젝트 클론 ===
import os, sys, json, time, re
import torch
import numpy as np
import pandas as pd
import wandb
from datetime import datetime
from typing import List, Dict, Optional, Any
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from tqdm.auto import tqdm

# --- 프로젝트 클론 ---
if not os.path.exists('/content/GovOn'):
    !git clone https://github.com/GovOn-Org/GovOn.git /content/GovOn
os.chdir('/content/GovOn')
PROJECT_ROOT = '/content/GovOn'

# --- GPU 확인 ---
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'VRAM: {vram_total:.1f} GB')
else:
    raise RuntimeError("GPU가 필요합니다. [런타임] -> [런타임 유형 변경] -> GPU 선택")

# --- W&B 로그인 ---
try:
    from google.colab import userdata
    WANDB_API_KEY = userdata.get('WANDB_API_KEY')
    print("WANDB_API_KEY: Colab Secrets에서 로드됨")
except (ImportError, Exception):
    WANDB_API_KEY = None
    print("WANDB_API_KEY: Colab Secrets 사용 불가")

if not WANDB_API_KEY:
    # WANDB_API_KEY = "your_wandb_key_here"  # 직접 입력 시 주석 해제
    pass

if WANDB_API_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_API_KEY
    wandb.login(key=WANDB_API_KEY, relogin=True)
    print('W&B logged in.')

print('\n환경 초기화 완료.')

In [ ]:
# === 1-C. 평가 설정 (Configuration) ===

# PRD v3.4 KPI 목표값
KPI_TARGETS = {
    "classification_accuracy": {"target": 85.0, "unit": "%", "label": "AC-001: 분류 정확도", "lower_is_better": False},
    "generation_p95_sec": {"target": 3.0, "unit": "초", "label": "AC-002: 답변 생성 p95", "lower_is_better": True},
    "retrieval_p95_sec": {"target": 1.0, "unit": "초", "label": "AC-003: 검색 p95", "lower_is_better": True},
    "vram_usage_gb": {"target": 5.0, "unit": "GB", "label": "VRAM 사용량", "lower_is_better": True},
}

CATEGORIES = ["교통", "행정", "환경", "세금", "기타", "건축", "복지", "안전"]


@dataclass
class KPIEvalConfig:
    """vLLM + AWQ KPI 평가 설정"""

    # --- 모델 ---
    awq_model_id: str = "umyunsang/GovOn-EXAONE-AWQ-v2"
    base_model_id: str = "LGAI-EXAONE/EXAONE-Deep-7.8B"

    # --- 데이터 ---
    test_data_path: str = ""

    # --- vLLM 엔진 설정 ---
    max_model_len: int = 4096
    gpu_memory_utilization: float = 0.80
    dtype: str = "float16"
    quantization: str = "awq"
    enforce_eager: bool = True
    trust_remote_code: bool = True

    # --- 생성 파라미터 ---
    max_new_tokens: int = 512
    temperature: float = 0.0
    top_p: float = 1.0
    repetition_penalty: float = 1.1

    # --- 시스템 메시지 (학습과 동일) ---
    system_message: str = "당신은 지자체 민원 담당 공무원을 돕는 AI 어시스턴트입니다."

    # --- FAISS / RAG ---
    embedding_model: str = "intfloat/multilingual-e5-large"
    faiss_data_path: str = ""
    faiss_index_path: str = ""
    retrieval_top_k: int = 5
    retrieval_n_queries: int = 200  # 검색 속도 측정용 쿼리 수

    # --- BERTScore ---
    bertscore_model: str = "bert-base-multilingual-cased"

    # --- W&B ---
    wandb_project: str = "umyun3/GovOn"
    wandb_run_name: str = ""
    wandb_tags: List[str] = field(default_factory=lambda: ["kpi-eval", "awq", "vllm", "M3"])

    # --- 출력 ---
    output_dir: str = ""

    def __post_init__(self):
        if not self.test_data_path:
            self.test_data_path = f"{PROJECT_ROOT}/data/processed/v2_test.jsonl"
        if not self.faiss_data_path:
            self.faiss_data_path = f"{PROJECT_ROOT}/data/processed/v2_train.jsonl"
        if not self.faiss_index_path:
            self.faiss_index_path = f"{PROJECT_ROOT}/models/faiss_index/complaints.index"
        if not self.output_dir:
            self.output_dir = f"{PROJECT_ROOT}/docs/outputs/kpi_evaluation"
        if not self.wandb_run_name:
            self.wandb_run_name = f"kpi-awq-vllm-{datetime.now().strftime('%m%d-%H%M')}"
        os.makedirs(self.output_dir, exist_ok=True)


config = KPIEvalConfig()

print("=== KPI 평가 설정 ===")
print(f"  AWQ 모델      : {config.awq_model_id}")
print(f"  테스트 데이터  : {config.test_data_path}")
print(f"  max_model_len  : {config.max_model_len}")
print(f"  GPU utilization: {config.gpu_memory_utilization}")
print(f"  max_new_tokens : {config.max_new_tokens}")
print(f"  W&B 프로젝트   : {config.wandb_project}")
print(f"  W&B 실행 이름  : {config.wandb_run_name}")

## 2. 평가 데이터셋 준비

v2_test.jsonl (1,265건, 8개 카테고리) 데이터를 로드하고 분포를 확인합니다.

**데이터 형식**: chat template 형식 (`[|system|]...[|user|]...[|assistant|]...`)
- `text`: 전체 대화 (시스템+사용자+어시스턴트)
- `category`: 정답 카테고리 (교통, 행정, 환경, 세금, 기타, 건축, 복지, 안전)
- `id`: 고유 식별자

In [ ]:
# === 2-A. 테스트 데이터 로드 ===

def parse_chat_template(text: str) -> dict:
    """EXAONE chat template에서 user 입력과 assistant 참조 답변을 추출"""
    result = {"system": "", "user_prompt": "", "complaint": "", "reference_answer": "", "gold_category": ""}

    # system 메시지 추출
    if "[|system|]" in text:
        sys_part = text.split("[|system|]")[1].split("[|endofturn|]")[0].strip()
        result["system"] = sys_part

    # user 메시지 추출
    if "[|user|]" in text:
        user_part = text.split("[|user|]")[1].split("[|endofturn|]")[0].strip()
        result["user_prompt"] = user_part

        # 카테고리 추출 (프롬프트에 포함된 [카테고리: XXX])
        cat_match = re.search(r'\[카테고리:\s*([^\]]+)\]', user_part)
        if cat_match:
            result["gold_category"] = cat_match.group(1).strip()

        # 민원 내용 추출
        if "민원 내용:" in user_part:
            complaint = user_part.split("민원 내용:")[1].strip()
            result["complaint"] = complaint

    # assistant 답변 추출
    if "[|assistant|]" in text:
        ans_part = text.split("[|assistant|]")[1]
        # endofturn이 있으면 거기까지, 없으면 전체
        if "[|endofturn|]" in ans_part:
            ans_part = ans_part.split("[|endofturn|]")[0]
        result["reference_answer"] = ans_part.strip()

    return result


def load_test_data(path: str) -> List[Dict]:
    """v2_test.jsonl 로드 및 파싱"""
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"WARNING: line {line_no} JSON 파싱 오류 스킵: {e}")
                continue

            parsed = parse_chat_template(item['text'])

            data.append({
                "id": item.get("id", f"sample_{line_no}"),
                "category": item.get("category", parsed["gold_category"]),
                "gold_category": parsed["gold_category"] or item.get("category", "unknown"),
                "complaint": parsed["complaint"],
                "user_prompt": parsed["user_prompt"],
                "reference_answer": parsed["reference_answer"],
                "system_message": parsed["system"],
                "raw_text": item['text'],
            })
    return data


test_data = load_test_data(config.test_data_path)
print(f"테스트 데이터 로드: {len(test_data)}건")
print(f"경로: {config.test_data_path}")

# 카테고리 분포
cat_counts = Counter(item["category"] for item in test_data)
print(f"\n카테고리 분포 (총 {len(cat_counts)}개):")
for cat, count in cat_counts.most_common():
    bar = "#" * (count // 5)
    print(f"  {cat:6s} : {count:5d} ({count / len(test_data) * 100:5.1f}%) {bar}")

# 동음이의어 테스트 케이스 확인
homonym_keywords = ["배수", "파손", "소음", "도로", "보도"]
homonym_count = sum(
    1 for item in test_data
    if any(kw in item["complaint"] for kw in homonym_keywords)
)
print(f"\n동음이의어 관련 키워드 포함 건수: {homonym_count}건")

# 참조 답변 길이 통계
ref_lengths = [len(item["reference_answer"]) for item in test_data]
print(f"\n참조 답변 길이 (chars):")
print(f"  평균={np.mean(ref_lengths):.0f}, 중앙값={np.median(ref_lengths):.0f}, "
      f"최소={min(ref_lengths)}, 최대={max(ref_lengths)}")

# 샘플 확인
print(f"\n--- 샘플 0 ---")
print(f"  ID       : {test_data[0]['id']}")
print(f"  카테고리 : {test_data[0]['category']}")
print(f"  민원(앞 100자): {test_data[0]['complaint'][:100]}...")
print(f"  답변(앞 100자): {test_data[0]['reference_answer'][:100]}...")

## 3. vLLM AWQ 모델 로드 및 GPU 메모리 모니터링

AWQ INT4 양자화 모델을 vLLM 엔진으로 로드하고 VRAM 사용량을 측정합니다.

In [ ]:
# === 3-A. GPU 메모리 모니터링 유틸리티 ===

def get_gpu_memory_info() -> dict:
    """현재 GPU 메모리 사용량 반환 (GB 단위)"""
    if not torch.cuda.is_available():
        return {"allocated_gb": 0, "reserved_gb": 0, "total_gb": 0, "free_gb": 0}

    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9

    # nvidia-smi 기반 실제 사용량 (더 정확)
    try:
        import subprocess
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True
        )
        parts = result.stdout.strip().split(',')
        used_mb = float(parts[0].strip())
        total_mb = float(parts[1].strip())
        return {
            "used_gb": used_mb / 1024,
            "total_gb": total_mb / 1024,
            "free_gb": (total_mb - used_mb) / 1024,
            "allocated_gb": allocated,
            "reserved_gb": reserved,
        }
    except Exception:
        return {
            "used_gb": allocated,
            "total_gb": total,
            "free_gb": total - allocated,
            "allocated_gb": allocated,
            "reserved_gb": reserved,
        }


# 모델 로드 전 GPU 메모리
pre_load_mem = get_gpu_memory_info()
print(f"모델 로드 전 GPU 메모리: {pre_load_mem['used_gb']:.2f} GB / {pre_load_mem['total_gb']:.1f} GB")

In [ ]:
# === 3-B. vLLM 엔진 로드 (AWQ 양자화) ===
from vllm import LLM, SamplingParams

print(f"vLLM 엔진 로드 시작: {config.awq_model_id}")
print(f"  quantization       : {config.quantization}")
print(f"  max_model_len      : {config.max_model_len}")
print(f"  gpu_memory_util    : {config.gpu_memory_utilization}")
print(f"  dtype              : {config.dtype}")
print(f"  enforce_eager      : {config.enforce_eager}")

load_start = time.perf_counter()

llm_engine = LLM(
    model=config.awq_model_id,
    trust_remote_code=config.trust_remote_code,
    max_model_len=config.max_model_len,
    gpu_memory_utilization=config.gpu_memory_utilization,
    dtype=config.dtype,
    quantization=config.quantization,
    enforce_eager=config.enforce_eager,
)

load_time = time.perf_counter() - load_start
tokenizer = llm_engine.get_tokenizer()

# 모델 로드 후 GPU 메모리
post_load_mem = get_gpu_memory_info()
model_vram = post_load_mem['used_gb'] - pre_load_mem['used_gb']

print(f"\nvLLM 엔진 로드 완료! ({load_time:.1f}초)")
print(f"  모델 VRAM 사용량: {model_vram:.2f} GB")
print(f"  전체 GPU 사용량 : {post_load_mem['used_gb']:.2f} GB / {post_load_mem['total_gb']:.1f} GB")
print(f"  Tokenizer vocab : {tokenizer.vocab_size}")

# Sanity check: 1건 생성 테스트
sanity_prompt = f"[|system|]{config.system_message}[|endofturn|]\n[|user|]안녕하세요, 테스트 입력입니다.[|endofturn|]\n[|assistant|]"
sanity_params = SamplingParams(temperature=0.0, max_tokens=50)
sanity_output = llm_engine.generate([sanity_prompt], sanity_params)
print(f"\nSanity check 출력: {sanity_output[0].outputs[0].text[:100]}")

## 4. AC-001: 분류 정확도 평가

**목표**: 테스트 데이터 기준 분류 정확도 >= 85%

**방법**: 민원 텍스트를 모델에 입력하여 카테고리를 분류하도록 하고, 정답 카테고리와 비교합니다.
분류 전용 프롬프트를 사용하여 모델이 카테고리만 출력하도록 유도합니다.

In [ ]:
# === 4-A. 분류 프롬프트 구성 ===

CLASSIFICATION_SYSTEM = "당신은 지자체 민원 분류 전문가입니다."

CLASSIFICATION_TEMPLATE = """다음 민원을 아래 카테고리 중 하나로 분류하세요.

카테고리: 교통, 행정, 환경, 세금, 기타, 건축, 복지, 안전

민원 내용:
{complaint}

반드시 카테고리 이름만 한 단어로 답하세요."""


def build_classification_prompt(complaint: str) -> str:
    """분류 전용 프롬프트 생성"""
    user_content = CLASSIFICATION_TEMPLATE.format(complaint=complaint[:1500])  # 길이 제한
    return (
        f"[|system|]{CLASSIFICATION_SYSTEM}[|endofturn|]\n"
        f"[|user|]{user_content}[|endofturn|]\n"
        f"[|assistant|]"
    )


def extract_predicted_category(output_text: str) -> str:
    """모델 출력에서 카테고리를 추출"""
    text = output_text.strip()

    # thought 태그 제거
    text = re.sub(r'<thought>.*?</thought>', '', text, flags=re.DOTALL).strip()

    # 직접 매칭: 카테고리 이름이 출력 첫 부분에 있는 경우
    for cat in CATEGORIES:
        if text.startswith(cat) or text == cat:
            return cat

    # 출력에서 카테고리 키워드 검색
    for cat in CATEGORIES:
        if cat in text:
            return cat

    # 유사 표현 매핑
    aliases = {
        "도로": "교통", "교통사고": "교통", "신호": "교통", "주차": "교통",
        "세무": "세금", "납세": "세금", "과세": "세금", "지방세": "세금",
        "쓰레기": "환경", "소음": "환경", "악취": "환경", "미세먼지": "환경", "수질": "환경",
        "허가": "건축", "건설": "건축", "인허가": "건축",
        "사회복지": "복지", "장애인": "복지", "보육": "복지",
        "재난": "안전", "화재": "안전", "사고": "안전",
        "민원": "행정", "공무원": "행정", "서류": "행정", "신고": "행정",
    }
    for alias, cat in aliases.items():
        if alias in text:
            return cat

    return "unknown"


# 프롬프트 샘플 확인
sample_prompt = build_classification_prompt(test_data[0]["complaint"])
print("=== 분류 프롬프트 샘플 ===")
print(sample_prompt[:500])
print("...")

In [ ]:
# === 4-B. 분류 실행 (vLLM 배치 추론) ===

# 분류 프롬프트 생성
classification_prompts = [
    build_classification_prompt(item["complaint"])
    for item in test_data
]
print(f"분류 프롬프트 생성 완료: {len(classification_prompts)}건")

# 분류용 SamplingParams (짧은 출력, greedy)
cls_params = SamplingParams(
    temperature=0.0,
    max_tokens=30,  # 카테고리 이름만 필요
    repetition_penalty=1.0,
    stop=["[|user|]", "[|system|]", "[|endofturn|]", "</s>", "<|endoftext|>", "\n"],
)

print("분류 추론 시작...")
cls_start = time.perf_counter()
cls_outputs = llm_engine.generate(classification_prompts, cls_params)
cls_total_time = time.perf_counter() - cls_start

# 결과 추출
predicted_categories = []
cls_raw_outputs = []
for output in cls_outputs:
    raw = output.outputs[0].text
    cls_raw_outputs.append(raw)
    predicted_categories.append(extract_predicted_category(raw))

print(f"분류 완료: {cls_total_time:.1f}초 ({len(test_data) / cls_total_time:.1f} samples/sec)")
print(f"\n분류 결과 예시 (처음 10건):")
for i in range(min(10, len(test_data))):
    match = "O" if predicted_categories[i] == test_data[i]["category"] else "X"
    print(f"  [{match}] 정답={test_data[i]['category']:4s} | 예측={predicted_categories[i]:7s} | 출력: {cls_raw_outputs[i][:40]}")

In [ ]:
# === 4-C. 분류 정확도 계산 및 분석 ===

gold_categories = [item["category"] for item in test_data]

# 전체 정확도
correct = sum(1 for g, p in zip(gold_categories, predicted_categories) if g == p)
total = len(gold_categories)
accuracy = correct / total * 100

print("=" * 70)
print(f"AC-001: 분류 정확도 = {accuracy:.2f}% ({correct}/{total})")
print(f"  목표: >= 85%  |  판정: {'PASS' if accuracy >= 85 else 'FAIL'}")
print("=" * 70)

# 카테고리별 정확도
print(f"\n카테고리별 정확도:")
cat_correct = defaultdict(int)
cat_total = defaultdict(int)
for g, p in zip(gold_categories, predicted_categories):
    cat_total[g] += 1
    if g == p:
        cat_correct[g] += 1

cat_accuracy_rows = []
for cat in sorted(cat_total.keys()):
    n = cat_total[cat]
    c = cat_correct[cat]
    acc = c / n * 100 if n > 0 else 0
    cat_accuracy_rows.append({"카테고리": cat, "정답": c, "전체": n, "정확도(%)": acc})
    status = "PASS" if acc >= 85 else "FAIL"
    print(f"  {cat:6s}: {acc:6.1f}% ({c:3d}/{n:3d}) [{status}]")

cat_accuracy_df = pd.DataFrame(cat_accuracy_rows)

# Confusion matrix (간략)
print(f"\n오분류 패턴 (상위 10건):")
confusion = defaultdict(int)
for g, p in zip(gold_categories, predicted_categories):
    if g != p:
        confusion[(g, p)] += 1

for (g, p), count in sorted(confusion.items(), key=lambda x: -x[1])[:10]:
    print(f"  {g:6s} -> {p:7s} : {count}건")

# unknown 예측 비율
unknown_count = predicted_categories.count("unknown")
print(f"\n'unknown' 예측 건수: {unknown_count}건 ({unknown_count / total * 100:.1f}%)")
if unknown_count > 0:
    print("  (모델이 카테고리를 명확히 출력하지 않은 경우)")
    # unknown 샘플 확인
    for i, (g, p, raw) in enumerate(zip(gold_categories, predicted_categories, cls_raw_outputs)):
        if p == "unknown" and i < 5:
            print(f"  샘플: 정답={g}, 출력='{raw[:60]}'")

## 5. AC-002: 답변 생성 속도 평가

**목표**: p95 < 3초 (GPU 환경)

**방법**: 테스트 데이터에서 샘플을 추출하여 개별 응답 시간을 측정하고, p50/p95/p99 분포를 계산합니다.
vLLM 배치 추론 대신 **개별 요청 단위**로 측정하여 실제 서빙 환경의 레이턴시를 반영합니다.

In [ ]:
# === 5-A. 답변 생성 프롬프트 및 속도 측정 ===

def build_generation_prompt(item: dict) -> str:
    """답변 생성용 프롬프트 (학습 시 사용한 chat template과 동일)"""
    # raw_text에서 assistant 답변 이전까지를 프롬프트로 사용
    text = item["raw_text"]
    if "[|assistant|]" in text:
        prompt = text.split("[|assistant|]")[0] + "[|assistant|]"
        return prompt
    # fallback
    return (
        f"[|system|]{config.system_message}[|endofturn|]\n"
        f"[|user|]{item['user_prompt']}[|endofturn|]\n"
        f"[|assistant|]"
    )


# 답변 생성용 SamplingParams
gen_params = SamplingParams(
    temperature=config.temperature,
    top_p=config.top_p,
    max_tokens=config.max_new_tokens,
    repetition_penalty=config.repetition_penalty,
    stop=["[|user|]", "[|system|]", "[|endofturn|]", "</s>", "<|endoftext|>"],
)

# 개별 레이턴시 측정 (카테고리별 균등 샘플링, 최소 200건)
N_LATENCY_SAMPLES = min(200, len(test_data))

# 카테고리별 균등 샘플링
latency_indices = []
samples_per_cat = max(1, N_LATENCY_SAMPLES // len(cat_counts))
for cat in cat_counts:
    cat_indices = [i for i, item in enumerate(test_data) if item["category"] == cat]
    np.random.seed(42)
    selected = np.random.choice(cat_indices, size=min(samples_per_cat, len(cat_indices)), replace=False)
    latency_indices.extend(selected.tolist())
latency_indices = latency_indices[:N_LATENCY_SAMPLES]

print(f"레이턴시 측정 샘플: {len(latency_indices)}건")
print("개별 요청 단위로 응답 시간 측정 중...")

# 개별 요청 측정
individual_latencies = []
generation_results = []

for idx in tqdm(latency_indices, desc="답변 생성 레이턴시 측정"):
    item = test_data[idx]
    prompt = build_generation_prompt(item)

    start = time.perf_counter()
    output = llm_engine.generate([prompt], gen_params)
    elapsed = time.perf_counter() - start

    gen_text = output[0].outputs[0].text
    gen_tokens = len(output[0].outputs[0].token_ids)

    individual_latencies.append(elapsed)
    generation_results.append({
        "idx": idx,
        "category": item["category"],
        "latency_sec": elapsed,
        "gen_tokens": gen_tokens,
        "gen_text": gen_text,
        "reference": item["reference_answer"],
    })

print(f"\n개별 레이턴시 측정 완료: {len(individual_latencies)}건")

In [ ]:
# === 5-B. 답변 생성 속도 분석 ===

latencies_arr = np.array(individual_latencies)

gen_p50 = np.percentile(latencies_arr, 50)
gen_p95 = np.percentile(latencies_arr, 95)
gen_p99 = np.percentile(latencies_arr, 99)
gen_mean = np.mean(latencies_arr)
gen_std = np.std(latencies_arr)
gen_min = np.min(latencies_arr)
gen_max = np.max(latencies_arr)

# 토큰 처리량
gen_tokens_arr = np.array([r["gen_tokens"] for r in generation_results])
tokens_per_sec = gen_tokens_arr / latencies_arr
avg_tps = np.mean(tokens_per_sec)

print("=" * 70)
print(f"AC-002: 답변 생성 속도")
print("=" * 70)
print(f"  p50       : {gen_p50:.3f}초")
print(f"  p95       : {gen_p95:.3f}초  (목표: < 3.0초, {'PASS' if gen_p95 < 3.0 else 'FAIL'})")
print(f"  p99       : {gen_p99:.3f}초")
print(f"  평균      : {gen_mean:.3f}초 (+/- {gen_std:.3f})")
print(f"  최소/최대 : {gen_min:.3f} / {gen_max:.3f}초")
print(f"  처리량    : {avg_tps:.1f} tokens/sec (평균)")
print(f"  평균 토큰 : {np.mean(gen_tokens_arr):.0f} tokens")
print("=" * 70)

# 레이턴시 분포 히스토그램 (텍스트 기반)
print("\n레이턴시 분포:")
bins = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 10.0, float('inf')]
labels = ["<0.5s", "0.5-1s", "1-1.5s", "1.5-2s", "2-2.5s", "2.5-3s", "3-4s", "4-5s", "5-10s", ">10s"]
for i in range(len(bins) - 1):
    count = np.sum((latencies_arr >= bins[i]) & (latencies_arr < bins[i+1]))
    pct = count / len(latencies_arr) * 100
    bar = "#" * int(pct / 2)
    print(f"  {labels[i]:8s}: {count:4d} ({pct:5.1f}%) {bar}")

## 6. AC-003: 유사 사례 검색 속도 평가

**목표**: FAISS 벡터 DB 검색 p95 < 1초

**방법**: multilingual-e5-large 임베딩 + FAISS IndexFlatIP로 유사 민원 검색 시간을 측정합니다.

In [ ]:
# === 6-A. FAISS 인덱스 구축 및 검색 속도 측정 ===
import faiss
from sentence_transformers import SentenceTransformer

print(f"임베딩 모델 로드: {config.embedding_model}")
embed_model = SentenceTransformer(config.embedding_model)

# FAISS 인덱스 로드 또는 구축
faiss_metadata = []

if os.path.exists(config.faiss_index_path):
    print(f"기존 인덱스 로드: {config.faiss_index_path}")
    faiss_index = faiss.read_index(config.faiss_index_path)
    meta_path = config.faiss_index_path + ".meta.json"
    if os.path.exists(meta_path):
        with open(meta_path, 'r', encoding='utf-8') as f:
            faiss_metadata = json.load(f)
    print(f"  인덱스 크기: {faiss_index.ntotal}건")
else:
    print(f"인덱스 구축 중: {config.faiss_data_path}")
    complaints = []
    faiss_metadata = []

    with open(config.faiss_data_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                item = json.loads(line)
                text = item['text']
                # 민원 내용 추출
                complaint = ""
                if "[|user|]" in text:
                    user_part = text.split("[|user|]")[1].split("[|endofturn|]")[0]
                    if "민원 내용:" in user_part:
                        complaint = user_part.split("민원 내용:")[1].strip()
                    else:
                        complaint = user_part.strip()

                answer = ""
                if "[|assistant|]" in text:
                    answer = text.split("[|assistant|]")[1].split("[|endofturn|]")[0].strip()

                complaints.append(f"passage: {complaint}")
                faiss_metadata.append({
                    "id": item.get("id"),
                    "category": item.get("category"),
                    "complaint": complaint,
                    "answer": answer,
                })
            except Exception as e:
                continue

    print(f"  문서 수: {len(complaints)}")
    print("  임베딩 인코딩 중...")
    embeddings = embed_model.encode(complaints, show_progress_bar=True, normalize_embeddings=True)

    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)
    faiss_index.add(embeddings.astype('float32'))

    # 저장
    os.makedirs(os.path.dirname(config.faiss_index_path), exist_ok=True)
    faiss.write_index(faiss_index, config.faiss_index_path)
    with open(config.faiss_index_path + ".meta.json", 'w', encoding='utf-8') as f:
        json.dump(faiss_metadata, f, ensure_ascii=False, indent=2)
    print(f"  인덱스 저장 완료: {config.faiss_index_path}")

print(f"\nFAISS 인덱스 준비 완료: {faiss_index.ntotal}건")

In [ ]:
# === 6-B. 검색 레이턴시 측정 ===

# 테스트 데이터에서 쿼리 샘플 추출
n_queries = min(config.retrieval_n_queries, len(test_data))
np.random.seed(42)
query_indices = np.random.choice(len(test_data), size=n_queries, replace=False)

retrieval_latencies = []
retrieval_results = []

print(f"검색 레이턴시 측정: {n_queries}건 (top_k={config.retrieval_top_k})")

for idx in tqdm(query_indices, desc="FAISS 검색"):
    query_text = test_data[idx]["complaint"]

    # E2E: 임베딩 + 검색 시간 모두 측정 (실제 서빙과 동일)
    start = time.perf_counter()

    # 1. 쿼리 임베딩
    query_emb = embed_model.encode(
        [f"query: {query_text}"], normalize_embeddings=True
    )

    # 2. FAISS 검색
    distances, indices = faiss_index.search(
        query_emb.astype('float32'), config.retrieval_top_k
    )

    elapsed = time.perf_counter() - start
    retrieval_latencies.append(elapsed)

    # 검색 결과 저장 (첫 번째 결과만)
    top_result = None
    if len(indices[0]) > 0 and indices[0][0] != -1 and indices[0][0] < len(faiss_metadata):
        top_result = faiss_metadata[indices[0][0]]

    retrieval_results.append({
        "query_idx": int(idx),
        "query_category": test_data[idx]["category"],
        "latency_sec": elapsed,
        "top1_score": float(distances[0][0]) if len(distances[0]) > 0 else 0,
        "top1_category": top_result["category"] if top_result else "N/A",
    })

# 결과 분석
ret_arr = np.array(retrieval_latencies)
ret_p50 = np.percentile(ret_arr, 50)
ret_p95 = np.percentile(ret_arr, 95)
ret_p99 = np.percentile(ret_arr, 99)

print("\n" + "=" * 70)
print(f"AC-003: 유사 사례 검색 속도")
print("=" * 70)
print(f"  p50       : {ret_p50*1000:.1f}ms ({ret_p50:.4f}초)")
print(f"  p95       : {ret_p95*1000:.1f}ms ({ret_p95:.4f}초)  (목표: < 1.0초, {'PASS' if ret_p95 < 1.0 else 'FAIL'})")
print(f"  p99       : {ret_p99*1000:.1f}ms ({ret_p99:.4f}초)")
print(f"  평균      : {np.mean(ret_arr)*1000:.1f}ms")
print(f"  최소/최대 : {np.min(ret_arr)*1000:.1f}ms / {np.max(ret_arr)*1000:.1f}ms")
print("=" * 70)

# Recall@5: 검색된 top-1 카테고리가 쿼리와 일치하는 비율
category_match = sum(
    1 for r in retrieval_results if r["top1_category"] == r["query_category"]
)
recall_at1 = category_match / len(retrieval_results) * 100
print(f"\n카테고리 기반 Recall@1: {recall_at1:.1f}%")
print(f"  (검색 결과 top-1의 카테고리가 쿼리 카테고리와 일치하는 비율)")

## 7. 답변 품질 평가 (BLEU, ROUGE-L, BERTScore)

레이턴시 측정 시 생성된 답변을 대상으로 표준 NLG 메트릭을 계산합니다.

In [ ]:
# === 7-A. 전처리 함수 ===
import sacrebleu
from rouge_score import rouge_scorer
import bert_score

# PII 토큰 패턴
_PII_PATTERNS = [
    r'▲+', r'\[NAME_MASKED\]', r'\[이름\]', r'\[전화번호\]', r'\[주소\]',
    r'<NAME>', r'<MOBILE_NUMBER>', r'<PHONE_NUMBER>', r'<DATE>',
    r'<BIRTH_NUMBER>', r'<ACCOUNT_NUMBER>', r'<ADDRESS>',
]
_PII_REGEX = re.compile('|'.join(_PII_PATTERNS))
_THOUGHT_REGEX = re.compile(r'<thought>.*?</thought>', flags=re.DOTALL)


def normalize_light(text: str) -> str:
    """경량 정규화: SacreBLEU용 (문장부호 유지)"""
    text = _THOUGHT_REGEX.sub('', text)
    text = _PII_REGEX.sub('', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def normalize_text(text: str) -> str:
    """풀 정규화: ROUGE-L / BERTScore용"""
    text = _THOUGHT_REGEX.sub('', text)
    text = _PII_REGEX.sub('', text)
    text = re.sub(r'[^가-힣a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


print("전처리 함수 정의 완료.")

In [ ]:
# === 7-B. 전체 테스트 데이터 배치 생성 (품질 평가용) ===
# 레이턴시 측정에서는 200건만 사용했으므로, 전체 데이터에 대해 배치 생성 실행

print(f"전체 테스트 데이터 배치 생성: {len(test_data)}건")

all_prompts = [build_generation_prompt(item) for item in test_data]

batch_start = time.perf_counter()
all_outputs = llm_engine.generate(all_prompts, gen_params)
batch_time = time.perf_counter() - batch_start

all_generations = [output.outputs[0].text for output in all_outputs]
all_references = [item["reference_answer"] for item in test_data]

print(f"배치 생성 완료: {batch_time:.1f}초 (처리량: {len(test_data)/batch_time:.1f} samples/sec)")

# GPU 메모리 (추론 중 피크)
inference_mem = get_gpu_memory_info()
print(f"추론 중 GPU 사용량: {inference_mem['used_gb']:.2f} GB")

In [ ]:
# === 7-C. NLG 메트릭 계산 ===

# SacreBLEU: 경량 정규화
light_hyps = [normalize_light(h) or "(empty)" for h in all_generations]
light_refs = [normalize_light(r) or "(empty)" for r in all_references]

# ROUGE-L, BERTScore: 풀 정규화
norm_hyps = [normalize_text(h) or "(empty)" for h in all_generations]
norm_refs = [normalize_text(r) or "(empty)" for r in all_references]

# 1. SacreBLEU
print("SacreBLEU 계산 중 (intl tokenizer)...")
bleu = sacrebleu.corpus_bleu(light_hyps, [light_refs], tokenize='intl')
bleu_score = bleu.score
bleu_bp = bleu.bp

# 2. ROUGE-L
print("ROUGE-L 계산 중...")
r_scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
rouge_scores = [r_scorer.score(r, h)["rougeL"].fmeasure for r, h in zip(norm_refs, norm_hyps)]
rouge_l_mean = np.mean(rouge_scores) * 100

# 3. BERTScore
print("BERTScore 계산 중...")
P, R, F1 = bert_score.score(
    norm_hyps, norm_refs,
    model_type=config.bertscore_model,
    lang="ko", verbose=True,
)
bertscore_f1_mean = F1.mean().item() * 100

# 길이 통계
gen_lengths = [len(h) for h in light_hyps]
ref_lengths_measured = [len(r) for r in light_refs]
length_ratio = np.mean(gen_lengths) / max(np.mean(ref_lengths_measured), 1)

# EOS 정상 종료율
eos_count = sum(1 for o in all_outputs if len(o.outputs[0].token_ids) < config.max_new_tokens)
eos_rate = eos_count / len(all_outputs) * 100

print("\n" + "=" * 70)
print("답변 품질 메트릭")
print("=" * 70)
print(f"  SacreBLEU      : {bleu_score:.2f} (BP={bleu_bp:.3f})")
print(f"  ROUGE-L F1     : {rouge_l_mean:.2f}")
print(f"  BERTScore F1   : {bertscore_f1_mean:.2f}")
print(f"  EOS 정상 종료  : {eos_rate:.1f}% ({eos_count}/{len(all_outputs)})")
print(f"  생성 길이 비율 : {length_ratio:.2f} (생성/참조)")
print(f"  평균 생성 길이 : {np.mean(gen_lengths):.0f} chars")
print(f"  평균 참조 길이 : {np.mean(ref_lengths_measured):.0f} chars")
print("=" * 70)

## 8. 종합 KPI 달성도 확인

PRD v3.4 수락 기준 대비 달성도를 종합 판정합니다.

In [ ]:
# === 8. 종합 KPI 달성도 테이블 ===

# 측정값 수집
measured_values = {
    "classification_accuracy": accuracy,
    "generation_p95_sec": gen_p95,
    "retrieval_p95_sec": ret_p95,
    "vram_usage_gb": inference_mem['used_gb'],
}

# KPI 판정
print("\n" + "=" * 80)
print("  PRD v3.4 KPI 종합 달성도 (M3 Phase)")
print("=" * 80)
print(f"{'KPI':30s} {'측정값':>12s} {'목표값':>12s} {'단위':>6s} {'판정':>8s}")
print("-" * 80)

all_passed = True
kpi_results = []

for key, info in KPI_TARGETS.items():
    value = measured_values.get(key, 0)
    target = info["target"]
    lower_better = info.get("lower_is_better", False)

    if lower_better:
        passed = value <= target
    else:
        passed = value >= target

    status = "PASS" if passed else "FAIL"
    if not passed:
        all_passed = False

    print(f"  {info['label']:28s} {value:>12.2f} {target:>12.1f} {info['unit']:>6s} {status:>8s}")

    kpi_results.append({
        "kpi": info["label"],
        "measured": value,
        "target": target,
        "unit": info["unit"],
        "passed": passed,
    })

print("-" * 80)

# 추가 메트릭 (KPI 외)
print(f"\n  {'[참고] 답변 품질 메트릭':28s}")
print(f"  {'SacreBLEU':28s} {bleu_score:>12.2f}")
print(f"  {'ROUGE-L F1':28s} {rouge_l_mean:>12.2f}")
print(f"  {'BERTScore F1':28s} {bertscore_f1_mean:>12.2f}")
print(f"  {'EOS 정상 종료율':28s} {eos_rate:>12.1f} {'%':>6s}")
print(f"  {'답변 생성 p50':28s} {gen_p50:>12.3f} {'초':>6s}")
print(f"  {'답변 생성 p99':28s} {gen_p99:>12.3f} {'초':>6s}")
print(f"  {'검색 p50':28s} {ret_p50*1000:>12.1f} {'ms':>6s}")
print(f"  {'모델 VRAM (모델만)':28s} {model_vram:>12.2f} {'GB':>6s}")
print(f"  {'처리량 (tokens/sec)':28s} {avg_tps:>12.1f}")
print(f"  {'배치 처리량 (samples/sec)':28s} {len(test_data)/batch_time:>12.1f}")

print("\n" + "=" * 80)
overall = "ALL PASSED" if all_passed else "SOME FAILED"
print(f"  종합 판정: {overall}")
print("=" * 80)

## 9. W&B 로깅

모든 평가 지표를 W&B 프로젝트(`umyun3/GovOn`)에 기록합니다.

기록 항목:
- KPI 달성도 테이블
- 분류 정확도 (전체 + 카테고리별)
- 생성 속도 분포
- 검색 속도 분포
- 답변 품질 메트릭 (BLEU, ROUGE-L, BERTScore)
- 모델 메타데이터 (AWQ 설정, VRAM 등)

In [ ]:
# === 9-A. W&B 실행 초기화 ===

run = wandb.init(
    project=config.wandb_project.split("/")[-1],  # "GovOn"
    entity=config.wandb_project.split("/")[0] if "/" in config.wandb_project else None,  # "umyun3"
    name=config.wandb_run_name,
    tags=config.wandb_tags,
    config={
        # 모델 설정
        "model": config.awq_model_id,
        "base_model": config.base_model_id,
        "quantization": config.quantization,
        "dtype": config.dtype,
        "max_model_len": config.max_model_len,
        "gpu_memory_utilization": config.gpu_memory_utilization,
        # 생성 파라미터
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "top_p": config.top_p,
        "repetition_penalty": config.repetition_penalty,
        # 데이터
        "test_data_path": config.test_data_path,
        "n_test_samples": len(test_data),
        "n_categories": len(cat_counts),
        # RAG
        "embedding_model": config.embedding_model,
        "faiss_index_size": faiss_index.ntotal,
        "retrieval_top_k": config.retrieval_top_k,
        # 환경
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
        "vram_total_gb": post_load_mem['total_gb'],
        "evaluation_type": "kpi_evaluation_awq_vllm",
        "prd_version": "v3.4",
    },
)

print(f"W&B 실행: {run.name}")
print(f"URL: {run.url}")

In [ ]:
# === 9-B. KPI 메트릭 로깅 ===

# 1. KPI 스칼라 메트릭
wandb.log({
    # AC-001: 분류
    "kpi/classification_accuracy": accuracy,
    "kpi/classification_total": total,
    "kpi/classification_correct": correct,
    # AC-002: 생성 속도
    "kpi/generation_p50_sec": gen_p50,
    "kpi/generation_p95_sec": gen_p95,
    "kpi/generation_p99_sec": gen_p99,
    "kpi/generation_mean_sec": gen_mean,
    "kpi/generation_throughput_tps": avg_tps,
    # AC-003: 검색 속도
    "kpi/retrieval_p50_ms": ret_p50 * 1000,
    "kpi/retrieval_p95_ms": ret_p95 * 1000,
    "kpi/retrieval_p99_ms": ret_p99 * 1000,
    "kpi/retrieval_recall_at1": recall_at1,
    # 효율성
    "kpi/vram_model_gb": model_vram,
    "kpi/vram_inference_gb": inference_mem['used_gb'],
    # 품질 메트릭
    "quality/sacrebleu": bleu_score,
    "quality/sacrebleu_bp": bleu_bp,
    "quality/rouge_l_f1": rouge_l_mean,
    "quality/bertscore_f1": bertscore_f1_mean,
    "quality/eos_rate": eos_rate,
    "quality/length_ratio": length_ratio,
    "quality/avg_gen_length_chars": np.mean(gen_lengths),
    # 배치 성능
    "performance/batch_total_sec": batch_time,
    "performance/batch_throughput_sps": len(test_data) / batch_time,
    "performance/model_load_sec": load_time,
})

# 2. KPI 달성도 테이블
kpi_table = wandb.Table(
    columns=["KPI", "측정값", "목표값", "단위", "판정"],
    data=[[r["kpi"], r["measured"], r["target"], r["unit"], "PASS" if r["passed"] else "FAIL"] for r in kpi_results]
)
wandb.log({"kpi/summary_table": kpi_table})

# 3. 카테고리별 분류 정확도 테이블
cat_acc_table = wandb.Table(dataframe=cat_accuracy_df)
wandb.log({"classification/category_accuracy": cat_acc_table})

# 카테고리별 스칼라
for _, row in cat_accuracy_df.iterrows():
    wandb.run.summary[f"classification/{row['카테고리']}/accuracy"] = row["정확도(%)"]
    wandb.run.summary[f"classification/{row['카테고리']}/n_samples"] = row["전체"]

print("KPI 메트릭 W&B 로깅 완료.")

In [ ]:
# === 9-C. 레이턴시 히스토그램 및 샘플 테이블 ===

# 1. 생성 레이턴시 히스토그램
wandb.log({
    "latency/generation_histogram": wandb.Histogram(individual_latencies, num_bins=30),
    "latency/retrieval_histogram": wandb.Histogram([l * 1000 for l in retrieval_latencies], num_bins=30),
})

# 2. 분류 결과 전체 테이블
cls_table_data = []
for i in range(len(test_data)):
    cls_table_data.append([
        test_data[i]["id"],
        test_data[i]["category"],
        predicted_categories[i],
        test_data[i]["category"] == predicted_categories[i],
        cls_raw_outputs[i][:100],
        test_data[i]["complaint"][:200],
    ])

cls_table = wandb.Table(
    columns=["id", "gold_category", "predicted_category", "correct", "raw_output", "complaint"],
    data=cls_table_data,
)
wandb.log({"classification/full_results": cls_table})

# 3. 답변 생성 샘플 테이블 (처음 50건)
gen_sample_data = []
for i in range(min(50, len(test_data))):
    gen_sample_data.append([
        test_data[i]["id"],
        test_data[i]["category"],
        all_references[i][:500],
        all_generations[i][:500],
        rouge_scores[i] * 100 if i < len(rouge_scores) else 0,
        (F1[i].item() * 100) if i < len(F1) else 0,
        len(normalize_text(all_generations[i])),
    ])

gen_table = wandb.Table(
    columns=["id", "category", "reference", "generation", "rouge_l", "bertscore_f1", "gen_length"],
    data=gen_sample_data,
)
wandb.log({"quality/sample_generations": gen_table})

# 4. 오분류 상세 테이블
misclass_data = []
for i in range(len(test_data)):
    if test_data[i]["category"] != predicted_categories[i]:
        misclass_data.append([
            test_data[i]["id"],
            test_data[i]["category"],
            predicted_categories[i],
            cls_raw_outputs[i][:100],
            test_data[i]["complaint"][:300],
        ])

misclass_table = wandb.Table(
    columns=["id", "gold", "predicted", "raw_output", "complaint"],
    data=misclass_data[:200],  # 최대 200건
)
wandb.log({"classification/misclassified": misclass_table})

print(f"W&B 테이블 로깅 완료.")
print(f"  분류 결과: {len(cls_table_data)}건")
print(f"  오분류: {len(misclass_data)}건")
print(f"  생성 샘플: {len(gen_sample_data)}건")

In [ ]:
# === 9-D. W&B 실행 종료 ===

# Summary에 주요 지표 저장
wandb.run.summary["kpi/all_passed"] = all_passed
wandb.run.summary["kpi/classification_accuracy"] = accuracy
wandb.run.summary["kpi/generation_p95_sec"] = gen_p95
wandb.run.summary["kpi/retrieval_p95_sec"] = ret_p95
wandb.run.summary["kpi/vram_usage_gb"] = inference_mem['used_gb']
wandb.run.summary["quality/sacrebleu"] = bleu_score
wandb.run.summary["quality/rouge_l_f1"] = rouge_l_mean
wandb.run.summary["quality/bertscore_f1"] = bertscore_f1_mean

wandb.finish()
print("W&B 실행 종료.")

## 10. 결과 저장 및 내보내기

In [ ]:
# === 10-A. 평가 결과 JSON 저장 ===

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
result_filename = f"kpi_eval_awq_vllm_{timestamp}.json"
result_path = os.path.join(config.output_dir, result_filename)

export_data = {
    "metadata": {
        "timestamp": timestamp,
        "prd_version": "v3.4",
        "model": config.awq_model_id,
        "base_model": config.base_model_id,
        "quantization": config.quantization,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
        "n_test_samples": len(test_data),
        "evaluation_type": "kpi_evaluation_awq_vllm",
    },
    "kpi_results": {
        "all_passed": all_passed,
        "details": kpi_results,
    },
    "classification": {
        "accuracy": accuracy,
        "total": total,
        "correct": correct,
        "category_accuracy": cat_accuracy_df.to_dict(orient="records"),
        "confusion_top10": [
            {"gold": g, "predicted": p, "count": c}
            for (g, p), c in sorted(confusion.items(), key=lambda x: -x[1])[:10]
        ],
    },
    "generation_latency": {
        "n_samples": len(individual_latencies),
        "p50_sec": gen_p50,
        "p95_sec": gen_p95,
        "p99_sec": gen_p99,
        "mean_sec": gen_mean,
        "std_sec": gen_std,
        "throughput_tps": avg_tps,
    },
    "retrieval_latency": {
        "n_queries": len(retrieval_latencies),
        "p50_ms": ret_p50 * 1000,
        "p95_ms": ret_p95 * 1000,
        "p99_ms": ret_p99 * 1000,
        "mean_ms": np.mean(ret_arr) * 1000,
        "recall_at1": recall_at1,
        "index_size": faiss_index.ntotal,
    },
    "quality_metrics": {
        "sacrebleu": bleu_score,
        "sacrebleu_bp": bleu_bp,
        "rouge_l_f1": rouge_l_mean,
        "bertscore_f1": bertscore_f1_mean,
        "eos_rate": eos_rate,
        "length_ratio": length_ratio,
    },
    "resource_usage": {
        "model_vram_gb": model_vram,
        "inference_vram_gb": inference_mem['used_gb'],
        "total_vram_gb": post_load_mem['total_gb'],
        "model_load_time_sec": load_time,
        "batch_inference_time_sec": batch_time,
    },
    "generation_config": {
        "max_new_tokens": config.max_new_tokens,
        "temperature": config.temperature,
        "top_p": config.top_p,
        "repetition_penalty": config.repetition_penalty,
        "max_model_len": config.max_model_len,
        "gpu_memory_utilization": config.gpu_memory_utilization,
    },
}

with open(result_path, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, ensure_ascii=False, indent=2, default=str)

print(f"결과 저장: {result_path}")

# Colab 자동 다운로드
try:
    from google.colab import files
    files.download(result_path)
    print(f"다운로드 시작: {result_filename}")
except ImportError:
    print("Colab 환경이 아닙니다. 수동으로 파일을 복사하세요.")

In [ ]:
# === 10-B. 최종 요약 출력 ===

print("\n" + "=" * 80)
print("  M3 vLLM + AWQ KPI 평가 최종 요약")
print("=" * 80)

print(f"""
  모델          : {config.awq_model_id}
  양자화        : AWQ INT4 ({config.dtype})
  테스트 데이터 : {len(test_data)}건 ({len(cat_counts)}개 카테고리)
  GPU           : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}

  ┌─────────────────────────────────────────────────────────────────┐
  │  KPI 달성도                                                     │
  ├─────────────────────────────────────────────────────────────────┤
  │  AC-001 분류 정확도  : {accuracy:6.2f}% (목표 >= 85%)   {'PASS' if accuracy >= 85 else 'FAIL':>10s}  │
  │  AC-002 생성 p95     : {gen_p95:6.3f}s (목표 < 3.0s)   {'PASS' if gen_p95 < 3.0 else 'FAIL':>10s}  │
  │  AC-003 검색 p95     : {ret_p95*1000:6.1f}ms (목표 < 1000ms) {'PASS' if ret_p95 < 1.0 else 'FAIL':>5s}  │
  │  VRAM 사용량         : {inference_mem['used_gb']:6.2f}GB (목표 < 5.0GB)  {'PASS' if inference_mem['used_gb'] <= 5.0 else 'FAIL':>5s}  │
  ├─────────────────────────────────────────────────────────────────┤
  │  답변 품질                                                      │
  │  SacreBLEU : {bleu_score:6.2f}  ROUGE-L : {rouge_l_mean:6.2f}  BERTScore : {bertscore_f1_mean:6.2f}  │
  │  EOS 종료율: {eos_rate:5.1f}%  길이비율: {length_ratio:.2f}x                     │
  └─────────────────────────────────────────────────────────────────┘

  종합 판정: {'ALL PASSED' if all_passed else 'SOME FAILED'}
  결과 파일: {result_path}
""")
print("=" * 80)
print("파이프라인 완료.")